In [1]:
import requests
from bs4 import BeautifulSoup

def fetch_mesh_scope(mesh_id):
    url = f"https://meshb.nlm.nih.gov/record/ui?ui={mesh_id}"
    r = requests.get(url, timeout=10)
    if r.status_code != 200:
        return None

    soup = BeautifulSoup(r.text, "html.parser")
    scope = soup.find(id= "scopeNote").find('dd')
    print(scope)
    return scope.get_text(strip=True) if scope else None

print(fetch_mesh_scope("D012378"))


<dd>Substances used to destroy or inhibit the action of rats, mice, or other rodents.</dd>
Substances used to destroy or inhibit the action of rats, mice, or other rodents.


In [2]:
def get_smiles(cid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IsomericSmiles/JSON"
    r = requests.get(url, timeout=30)
    if not r.ok:
        return None
    props = r.json().get("PropertyTable", {}).get("Properties", [])
    if not props:
        return None
    return props[0].get("SMILES")
print(get_smiles(7565))

C1=CC=C(C=C1)OC2=CC=C(C=C2)Br


In [3]:
import requests

def get_targets_for_assay(aid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/targets/ProteinAccession/JSON"
    r = requests.get(url, timeout=30)
    if r.ok:
        return r.json().get("InformationList", [{}]).get('Information',[{}])[0].get('ProteinAccession')
    return []

def get_assays_for_cid(cid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/aids/JSON"
    r = requests.get(url, timeout=30)
    if r.ok:
        return r.json().get('InformationList',[{}]).get('Information',[{}])[0].get('AID')
    return []

def get_targets_for_cid(cid):
    targets = set()
    aids = get_assays_for_cid(cid)
    for aid in aids:
        posstarget = get_targets_for_assay(aid)
        for acc in posstarget:
            targets.add(acc)
    return list(targets)

In [4]:
import pandas as pd

chemicals = pd.read_csv('chemicalids.csv')
chemicals = chemicals[chemicals['type'] == 'c']
print(chemicals.head())


            chemID       cids type
1     MESH:C070055      35823    c
2     MESH:C024713      12389    c
3  MESH:D000077612     166548    c
4     MESH:C538809  131634859    c
5  MESH:C000626479       7565    c


In [5]:
import csv
from tqdm import tqdm
with open('chemds.csv',mode='w',newline="") as file:
    writer = csv.writer(file, delimiter='|')
    for chem in tqdm(chemicals.itertuples()):
        # des = fetch_mesh_scope(chem.chemID[5:])
        smiles = get_smiles(chem.cids)
        targets = get_targets_for_cid(chem.cids)
        writer.writerow([chem.chemID[5:],chem.cids,targets,smiles])
        file.flush()


138it [15:42:13, 409.66s/it] 


ConnectTimeout: HTTPSConnectionPool(host='pubchem.ncbi.nlm.nih.gov', port=443): Max retries exceeded with url: /rest/pug/assay/aid/1920063/targets/ProteinAccession/JSON (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000292DB103B10>, 'Connection to pubchem.ncbi.nlm.nih.gov timed out. (connect timeout=30)'))

In [2]:
import pandas as pd
data = pd.read_csv('chemsmiles.csv')
print(data.head())
print(data.info())

       meshID       cids                                             smiles
0     C070055      35823           C1=CC(=C(C=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl
1     C024713      12389                                     CCCCCCCCCCCCCC
2  D000077612     166548  CCCCCOC1=CC=C(C=C1)C2=CC=C(C=C2)C3=CC=C(C=C3)C...
3     C538809  131634859                  B(=O)OB1OB2OB(OB(O2)O1)[O-].[Na+]
4  C000626479       7565                      C1=CC=C(C=C1)OC2=CC=C(C=C2)Br
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13943 entries, 0 to 13942
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   meshID  13943 non-null  object
 1   cids    13943 non-null  int64 
 2   smiles  13943 non-null  object
dtypes: int64(1), object(2)
memory usage: 326.9+ KB
None


In [8]:
data['type'] = data['meshID'].str[:1]
print(data[data['type'] == 'D'].count())

meshID    2680
cids      2680
smiles    2680
type      2680
dtype: int64
